# Smoothed persistence, ruling out gbm

The capture diagnostic found gbm's one win, more PnL on the partial-adjust rule, was pure turnover: a smoother prediction series rebalances less, so it pays less in fees, while actually capturing less funding than persistence. That is not a forecast, it is smoothness, and smoothness does not need a model.

This tests it directly. Take persistence, the same one-print signal, and smooth it with an ewma of the same past-funding input it already uses. If that reproduces gbm's low turnover it reproduces gbm's only edge without training, features or a model, and gbm is ruled out. Same expanding monthly policy, same two rules, same cost.

## Setup and predictions

Same pipeline and causal expanding-monthly generator as the capture diagnostic. elastic and gbm come along as the reference sources.

In [1]:
import os, sys, glob, warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "research").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import research.funding_carry.features as F
import research.cv as CV
from research.funding_carry.models import _with_symbol_dummies

_so = glob.glob(str(REPO / "build" / "release" / "**" / "qp_python_backtest*.so"), recursive=True)
sys.path.insert(0, os.path.dirname(_so[0]))
import qp_python_backtest as qb

DATA       = REPO / "data" / "binance_historical"
COST_TABLE = str(REPO / "data" / "cost_model" / "cost_table.csv")
SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
           "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT", "LTCUSDT"]
NAME_TO_ID = {n: i for i, n in enumerate(SYMBOLS)}
ID_TO_NAME = {i: n for n, i in NAME_TO_ID.items()}
FUT, SPOT = 0, 1
HORIZON = 24
HURDLE = 0.0034

In [2]:
FUNDING_COLS = ["symbol", "tag", "ts_ms", "interval_h", "funding_rate"]
PREMIUM_COLS = ["ts_ms", "open", "high", "low", "close", "volume", "close_ms",
                "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]


def load_funding(sym):
    d = DATA / sym / "futures" / "funding"
    files = sorted(d.glob(f"{sym}-fundingRate-*.csv"))
    df = pd.concat([pd.read_csv(f, header=None, names=FUNDING_COLS) for f in files], ignore_index=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    df["symbol"] = sym
    return (df[["symbol", "ts", "funding_rate"]].rename(columns={"funding_rate": "realized_funding"})
            .sort_values("ts").drop_duplicates("ts").reset_index(drop=True))


def load_premium(sym):
    d = DATA / sym / "futures" / "premiumindex"
    files = sorted(d.glob("*.csv"))
    if not files:
        return pd.DataFrame(columns=["ts_ms", "close"])
    parts = [pd.read_csv(f, header=None, names=PREMIUM_COLS)[["ts_ms", "close"]] for f in files]
    df = pd.concat(parts, ignore_index=True).sort_values("ts_ms").drop_duplicates("ts_ms").reset_index(drop=True)
    df["ts"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True)
    return df


def average_premium_up_to(premium, funding_ts, interval_hours=8):
    if premium.empty:
        return pd.Series(np.nan, index=range(len(funding_ts)))
    p = premium.sort_values("ts").reset_index(drop=True)
    win = pd.Timedelta(hours=interval_hours)
    out = np.full(len(funding_ts), np.nan)
    ts_vals = pd.to_datetime(funding_ts, utc=True).to_numpy()
    p_ts = p["ts"].to_numpy(); p_close = p["close"].to_numpy(dtype=float)
    for i, end in enumerate(ts_vals):
        lo = np.searchsorted(p_ts, end - win, side="right")
        hi = np.searchsorted(p_ts, end, side="right")
        if hi > lo:
            out[i] = p_close[lo:hi].mean()
    return pd.Series(out)


parts = []
for sym in SYMBOLS:
    f = load_funding(sym); p = load_premium(sym)
    f["premium"] = average_premium_up_to(p, f["ts"]).to_numpy()
    parts.append(f)
events = pd.concat(parts, ignore_index=True)

feat = events.copy()
for k in (1, 2, 3):
    feat = F.add_funding_lag(feat, k)
feat = F.add_funding_ewma(feat, halflife=2); feat = F.add_funding_ewma(feat, halflife=6)
feat = F.add_funding_vol(feat, window=8); feat = F.add_clamp_distance(feat)
feat = F.add_premium_trend(feat, span=3); feat = F.add_cross_symbol_spread(feat, reference="BTCUSDT")
feat = F.add_basket_spread(feat); feat = F.add_time_features(feat)
feat = F.add_funding_mean_window(feat, window=90); feat = F.add_funding_sign_window(feat, window=90)
feat = F.add_funding_vol_rank(feat, vol_window=30, rank_window=180)
kline_agg = pd.read_pickle(REPO / "research" / "funding_carry" / "results" / "kline_aggregates_8h.pkl")
feat = feat.merge(kline_agg, on=["symbol", "ts"], how="left")
feat["taker_imb_x_ewma2"] = feat["taker_imbalance"] * feat["funding_ewma_h2"]
feat["taker_imb_x_lag1"] = feat["taker_imbalance"] * feat["funding_lag1"]
feat["vol1m_x_clamp"] = feat["realized_vol_1m"] * feat["clamp_distance"]
feat["range_x_ewma2"] = feat["high_low_range"] * feat["funding_ewma_h2"]
feat["regime_x_lag1"] = feat["funding_sign_w90"] * feat["funding_lag1"]
feat["basket_x_lag1"] = feat["basket_spread"] * feat["funding_lag1"]
feat["lag1_sq"] = feat["funding_lag1"] ** 2
feat = F.add_basket_zscore(feat, source="funding_lag1")
feat = F.add_basket_zscore(feat, source="funding_ewma_h6")
feat = F.add_basket_rank(feat, source="funding_lag1")
feat["basket_z_x_lag1"] = feat["basket_z_funding_lag1"] * feat["funding_lag1"]
feat["basket_rank_x_lag1"] = feat["basket_rank_funding_lag1"] * feat["funding_lag1"]
feat = F.add_cum_target(feat, horizon=HORIZON)
FCOLS = [c for c in feat.columns
         if c not in ("symbol", "ts", "realized_funding", "premium", "realized_cum")]
feat = feat.dropna(subset=FCOLS + ["realized_funding", "realized_cum"]).reset_index(drop=True)
feat["year"] = feat["ts"].dt.year
folds = list(CV.walk_forward_splits(feat, n_folds=5, horizon=HORIZON, embargo=5))
fold_of = np.full(len(feat), -1)
for fi, (_, te) in enumerate(folds):
    fold_of[te] = fi
feat["fold"] = fold_of


PURGE = pd.Timedelta(hours=HORIZON * 8)


def elastic_pred(train, test, cols):
    Xtr, _ = _with_symbol_dummies(train, cols)
    pipe = Pipeline([("sc", StandardScaler()),
                     ("m", ElasticNet(alpha=1e-4, l1_ratio=0.7, max_iter=20000))])
    pipe.fit(Xtr, train["realized_cum"].to_numpy())
    Xte, _ = _with_symbol_dummies(test, cols)
    return pipe.predict(Xte)


def gbm_pred(train, test, cols):
    import lightgbm as lgb
    Xtr, c = _with_symbol_dummies(train, cols)
    y = train["realized_cum"].to_numpy(); sp = int(len(y) * 0.8)
    p = dict(objective="regression", verbose=-1, feature_fraction=0.9, bagging_fraction=0.9,
             bagging_freq=5, learning_rate=0.02, num_leaves=15, min_data_in_leaf=500)
    dtr = lgb.Dataset(Xtr[:sp], label=y[:sp], feature_name=c)
    dval = lgb.Dataset(Xtr[sp:], label=y[sp:], reference=dtr)
    b = lgb.train(p, dtr, num_boost_round=3000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    Xte, _ = _with_symbol_dummies(test, cols)
    return b.predict(Xte, num_iteration=b.best_iteration)


MODELS = {"elastic": elastic_pred, "gbm": gbm_pred}


def gen_policy_oof(window_months, refit_freq, min_train=2000):
    test = feat[feat.fold >= 0]
    start = pd.Timestamp(test.ts.min()).tz_convert("UTC").normalize().replace(day=1)
    dates = pd.date_range(start, test.ts.max(), freq=refit_freq, tz="UTC")
    di = np.searchsorted(dates.to_numpy(), test.ts.to_numpy(), side="right") - 1
    out = test[["symbol", "ts", "fold", "year", "realized_funding",
                "realized_cum", "funding_lag1"]].copy()
    for name in MODELS:
        out["pred_" + name] = np.nan
    out["pred_persistence"] = (HORIZON * test["funding_lag1"]).to_numpy()
    for k, d in enumerate(dates):
        sl = test[di == k]
        if sl.empty:
            continue
        tr = feat[feat.ts <= d - PURGE]
        if window_months is not None:
            tr = tr[tr.ts >= d - pd.DateOffset(months=window_months)]
        if len(tr) < min_train:
            continue
        for name, fn in MODELS.items():
            out.loc[sl.index, "pred_" + name] = fn(tr, sl, FCOLS)
    return out


oof = gen_policy_oof(None, "MS").dropna(subset=["pred_elastic", "pred_gbm"])
oof = oof.sort_values(["symbol", "ts"]).reset_index(drop=True)
print("oof rows", len(oof))

oof rows 22448


## Smoothed persistence

Persistence is `24 * funding_lag1`, the last print extrapolated. Smoothed persistence replaces the single last print with an ewma of the same past-funding series, per symbol, over a halflife of 2 to 6 prints, still times 24. Same information set, strictly causal, just less jittery. No fit, no features, no model.

In [3]:
for h in [2, 4, 6]:
    oof[f"pred_psm{h}"] = HORIZON * oof.groupby("symbol").funding_lag1.transform(
        lambda s: s.ewm(halflife=h).mean())

SRC = ["persistence", "psm2", "psm4", "psm6", "gbm"]
avail = oof.realized_funding.clip(lower=0); A = avail.sum()
rows = []
for src in SRC:
    held = (oof["pred_" + src] >= HURDLE)
    rows.append({"source": src, "capture_ratio": round((held * oof.realized_funding).sum() / A, 3),
                 "hold_frac": round(held.mean(), 3)})
display(pd.DataFrame(rows).set_index("source"))

,capture_ratio,hold_frac
source,,
persistence,0.381,0.122
psm2,0.416,0.145
psm4,0.421,0.153
psm6,0.425,0.160
gbm,0.344,0.122


Smoothing raises capture. Halflife 6 captures 0.425 of the available funding against raw persistence's 0.381, by holding a little more, 0.16 against 0.12, riding through a single soft print instead of dropping out on it. gbm at 0.344 is below both. So the ewma keeps persistence's capture and improves it, the opposite of gbm, which smoothed by capturing less.

## Dollar runs, the two rules

Threshold and slow partial adjustment on each source. Partial adjustment is where turnover bites, so it is the decisive comparison against gbm.

In [4]:
class PredThreshold:
    def __init__(self, bt, pred, qty=1.0, enter=0.0034, exit=0.0):
        self.bt = bt; self.pred = pred; self.qty = qty; self.enter = enter; self.exit = exit
    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        held = self.bt.position(ev.symbol, SPOT)
        t = self.qty if p >= self.enter else (0.0 if p <= self.exit else held)
        return [qb.Intent(ev.symbol, SPOT, t), qb.Intent(ev.symbol, FUT, -t)]
    def on_timer(self, now):
        return None


class PredPartialAdjust:
    def __init__(self, bt, pred, hurdle=0.0034, cap=2.0, adjust=0.15):
        self.bt = bt; self.pred = pred; self.hurdle = hurdle; self.cap = cap; self.adjust = adjust
    def on_event(self, ev):
        if ev.kind != qb.EventKind.Funding or ev.venue != FUT:
            return None
        p = self.pred.get((ev.symbol, ev.ts))
        if p is None:
            return None
        aim = float(np.clip(p / self.hurdle, 0.0, self.cap))
        cur = self.bt.position(ev.symbol, SPOT)
        t = cur + self.adjust * (aim - cur)
        return [qb.Intent(ev.symbol, SPOT, t), qb.Intent(ev.symbol, FUT, -t)]
    def on_timer(self, now):
        return None


class Gate:
    def __init__(self, bt, cap=10.0):
        self.bt = bt; self.cap = cap; self._id = 1
    def _next(self):
        i = self._id; self._id += 1; return i
    def check(self, it):
        cur = self.bt.position(it.symbol, it.venue)
        t = float(np.clip(it.target_position, -self.cap, self.cap)); d = t - cur
        out = qb.RiskOutcome.Approved if t == it.target_position else qb.RiskOutcome.Resized
        side = qb.Side.Buy if d >= 0 else qb.Side.Sell
        return qb.RiskDecision(out, qb.Order(self._next(), it.symbol, side, it.venue, abs(d)))


PF = oof.ts.min().date().isoformat(); PL = oof.ts.max().date().isoformat()


def pred_map(col):
    d = oof.dropna(subset=[col])
    sid = d.symbol.map(NAME_TO_ID).to_numpy()
    tns = (d.ts.dt.tz_convert("UTC").dt.tz_localize(None)
           .values.astype("datetime64[ns]").astype("int64"))
    return dict(zip(zip(sid, tns), d[col]))


def run(col, mk):
    ds = qb.Dataset(str(DATA), SYMBOLS, PF, PL, COST_TABLE)
    bt = qb.PythonBacktest(ds)
    st = mk(bt, pred_map(col)); g = Gate(bt)
    bt.set_on_event(st.on_event); bt.set_on_timer(st.on_timer); bt.set_check(g.check)
    bt.set_event_kinds([qb.EventKind.Funding])
    r = bt.run()
    fn = float(np.sum(r.funding_received)) - float(np.sum(r.funding_paid))
    return {"funding_net": round(fn, 1), "fees": round(float(np.sum(r.fees)), 1),
            "fee_drag": round(float(np.sum(r.fees)) / fn, 3) if fn else np.nan,
            "final": round(r.final_equity, 1)}


rows = []
for sname, mk in [("threshold", lambda bt, p: PredThreshold(bt, p, 1.0, 0.0034, 0.0)),
                  ("partial", lambda bt, p: PredPartialAdjust(bt, p, 0.0034, 2.0, 0.15))]:
    for src in SRC:
        rows.append({"strategy": sname, "source": src, **run("pred_" + src, mk)})
res = pd.DataFrame(rows).set_index(["strategy", "source"])
display(res)

funding_net    fees  fee_drag    final
strategy  source                                             
threshold persistence       8684.3   380.1     0.044   8580.0
          psm2              9079.1   242.2     0.027   9121.3
          psm4              8865.9   155.8     0.018   8782.9
          psm6              8879.1   154.2     0.017   8882.2
          gbm               7783.2   155.2     0.020   7843.3
partial   persistence      11704.1  2426.8     0.207   9679.3
          psm2             11830.7  1564.6     0.132  10692.5
          psm4             11814.8  1235.0     0.105  11008.1
          psm6             11775.2  1046.4     0.089  11143.8
          gbm              11127.9  1101.0     0.099  10437.8

Smoothed persistence beats gbm on every axis. On partial adjustment halflife 6 nets 11144 against gbm's 10438, and it does it by collecting more gross funding, 11775 against 11128, at a lower fee drag, 0.089 against 0.099. Even halflife 2 clears gbm. More smoothing helps monotonically here, 10692 to 11144 from halflife 2 to 6, so it is a robust direction not a tuned point. On the threshold rule the same, every smoothed variant clears gbm's 7843 and raw persistence's 8580.

gbm's advantage was a smoother signal that churns less. An ewma of the funding input is smoother still and churns less, and unlike gbm it keeps the capture, because gbm bought its smoothness by regressing toward a flatter function that holds less of the funding. gbm is strictly dominated by a signal with no training, no features and no model.

## Read

gbm is ruled out. Its only win over persistence, more net PnL on partial adjustment, was turnover, not forecast, and a one-line ewma of the funding input reproduces the low turnover and then beats gbm outright on capture, fee drag and net. The smoothing is the edge, and the smoothing is free.

So the live signal for the funding carry is smoothed persistence, an ewma of the last few funding prints over a halflife around 6, on the slow partial-adjust rule. No model. The carry research closes here on the signal: the content is in recent funding, persistence holds it, smoothing harvests it cheaply, and ML adds nothing at the level. What remains is cost, the short side, and a risk or regime model whose target is book drawdown, never the funding level.